# Adaptive Multiscale Spectro-Topological (AMST) Shape Descriptor
## MPEG-7 CE-Shape-1 Evaluation — v12-FIXED2

**Critical fixes over v12:**
- **Spline-smoothed contours** (128 pts, subpixel accuracy) for clean curvature
- **5-fold CV** (was 3-fold) for more training data per fold
- **Fixed C2 topology**: Autocorrelation tau, z-score normalization, aggressive noise filtering
- **Fixed MHFisherAttn**: Deterministic stratified sampling, temperature-softened Fisher scoring
- **Expanded SVM grid**: 70x hyperparameter combinations
- **Improved C3 SPD**: 22 basis functions (was 20) for richer shape covariance
- **AMST-only attention**: Attention is part of AMST design, baselines compared without it

In [ ]:
# Cell 1: Install
!pip install -q PyWavelets ripser persim xgboost scikit-image scikit-learn matplotlib seaborn scipy numpy pandas tqdm

import os, sys, warnings, json, copy, re, gc
warnings.filterwarnings('ignore')
os.environ['OMP_NUM_THREADS'] = '4'
print('Done.')

In [ ]:
# Cell 2: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm
import scipy, scipy.stats, scipy.special
from scipy import ndimage
from scipy.interpolate import interp1d, splprep, splev
from scipy.spatial import ConvexHull

import pywt
print(f'PyWavelets: {pywt.__version__}')

from ripser import ripser as ripser_fn
RIPSER_OK = True
print('Ripser: OK')

try:
    import xgboost as xgb
    XGB_OK = True
    print(f'XGBoost: {xgb.__version__}')
except:
    XGB_OK = False
    print('XGBoost: not available')

from skimage import io, color, transform, feature, measure, img_as_float
from skimage.filters import threshold_otsu
from skimage.morphology import binary_closing, binary_opening, disk, remove_small_objects
from skimage.measure import find_contours, regionprops, label

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

SEED=42; AMST_DIM=525
np.random.seed(SEED)
N_SPLITS=5  # 5-fold for more training data
print(f'SciPy {scipy.__version__} | Seed={SEED} | Folds={N_SPLITS} | AMST={AMST_DIM}')

In [ ]:
# Cell 3: Load MPEG-7 CE-Shape-1
DATA = Path('/content/mpeg7_shapes')
if not DATA.exists():
    import zipfile
    z = Path('/content/MPEG7_CE-Shape-1_Part_B.zip')
    if z.exists():
        with zipfile.ZipFile(str(z), 'r') as zf: zf.extractall('/content/mpeg7_shapes')
    else:
        raise FileNotFoundError('Upload MPEG7_CE-Shape-1_Part_B.zip to /content/')

img_dir = DATA / 'MPEG7_CE-Shape-1_Part_B'
if not img_dir.exists(): img_dir = DATA
EXT = {'.gif','.png','.jpg'}
all_files = sorted([f for f in img_dir.rglob('*') if f.suffix.lower() in EXT])
all_files = [f for f in all_files if f.stem.lower() not in ('confusions','shapedata')]
print(f'Files: {len(all_files)}')

def parse_label(fp):
    s = fp.stem
    return re.sub(r'[-_]?\d+$', '', s).strip('-_').lower()

samples = [(f, parse_label(f)) for f in all_files]
classes = sorted(set(s[1] for s in samples))
print(f'Samples: {len(samples)} | Classes: {len(classes)}')
print(f'Classes: {classes}')

In [ ]:
# Cell 4: Preprocessing (spline-smoothed contours, 128 pts)
IMG_SIZE = (64, 64)
CONTOUR_PTS = 128  # Optimal for 64x64 images (avoids over-sampling pixelation)

def load_bin(path):
    img = io.imread(str(path))
    img = np.squeeze(img)
    if img.ndim == 3:
        gray = color.rgb2gray(img[...,:3] if img.shape[2]==4 else img)
    elif img.ndim == 2:
        gray = img_as_float(img)
    else:
        raise ValueError(f'Unexpected shape: {img.shape}')
    gray = transform.resize(gray, IMG_SIZE, anti_aliasing=True)
    try: t = threshold_otsu(gray)
    except: t = 0.5
    b = gray < t
    if b.sum() < IMG_SIZE[0]*IMG_SIZE[1]*0.02: b = ~b
    b = binary_closing(b, disk(2))
    b = binary_opening(b, disk(1))
    b = remove_small_objects(b.astype(bool), min_size=50)
    return b.astype(np.uint8)

def get_contour(b, n=CONTOUR_PTS):
    cl = find_contours(b.astype(float), 0.5)
    if not cl: return np.zeros((n,2))
    c = max(cl, key=len)
    # Spline smoothing for subpixel-accurate contour
    try:
        tck, u = splprep([c[:,0], c[:,1]], s=0, per=True)
        u_new = np.linspace(0, 1, n, endpoint=False)
        x_s, y_s = splev(u_new, tck)
        c = np.column_stack([x_s, y_s])
    except:
        # Fallback: arc-length resampling
        d = np.diff(c, axis=0)
        arc = np.r_[0, np.cumsum(np.sqrt((d**2).sum(axis=1)))]
        if arc[-1] < 1e-8: return np.zeros((n,2))
        u = np.linspace(0, arc[-1], n, endpoint=False)
        c = np.column_stack([np.interp(u, arc, c[:,0]), np.interp(u, arc, c[:,1])])
    return c

def center_scale(c):
    c = c - c.mean(axis=0)
    r = np.sqrt((c**2).sum(axis=1)).max()
    return c / r if r > 1e-8 else c

def curvature(c):
    x, y = c[:,1], c[:,0]
    x1=np.gradient(x); y1=np.gradient(y); x2=np.gradient(x1); y2=np.gradient(y1)
    return (x1*y2 - x2*y1) / (x1**2 + y1**2 + 1e-12)**1.5

print('Loading shapes...')
bins, cnts, kurvs, labs = [], [], [], []
for path, label in tqdm(samples):
    try:
        b = load_bin(path)
        c = center_scale(get_contour(b))
        k = curvature(c)
        bins.append(b); cnts.append(c); kurvs.append(k); labs.append(label)
    except Exception as e:
        print(f'Fail: {path.name} - {e}')

le = LabelEncoder()
y = le.fit_transform([str(l) for l in labs])
n_cls = len(le.classes_)
print(f'Loaded: {len(cnts)} | Classes: {n_cls}')

In [ ]:
# Figure 1: MPEG-7 Samples
n_show = min(15, n_cls)
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()
for i in range(n_show):
    idx = np.where(y==i)[0][0]
    axes[i].imshow(bins[idx], cmap='gray')
    axes[i].set_title(le.classes_[i].capitalize(), fontsize=8, fontweight='bold'); axes[i].axis('off')
for ax in axes[n_show:]: ax.axis('off')
plt.suptitle(f'Figure 1: MPEG-7 CE-Shape-1 ({len(cnts)} images, {n_cls} classes)', fontweight='bold')
plt.tight_layout(); plt.savefig('/content/fig1_samples.png', dpi=120, bbox_inches='tight')
plt.show(); print('Fig 1 saved.')

In [ ]:
# Cell 5: Baseline Descriptors
def fourier_desc(c, K=64):
    r = np.sqrt((c**2).sum(axis=1))
    F = np.fft.fft(r); mag = np.abs(F)
    d = mag[1] if mag[1]>1e-8 else mag.max()+1e-12
    mag_n = mag / d
    return np.concatenate([mag_n[1:K+1][:-1], np.angle(F)[1:17]])

def wavelet_desc(c):
    r = np.sqrt((c**2).sum(axis=1)); r -= r.mean()
    f = []
    for w in ['db4','haar','sym4']:
        lv = max(1, min(5, pywt.dwt_max_level(len(r), w)))
        coeffs = pywt.wavedec(r, w, level=lv, mode='periodization')
        en = np.array([np.sum(co**2) for co in coeffs]); en /= en.sum()+1e-12
        f.append(en)
    mx = max(len(e) for e in f)
    return np.concatenate([np.pad(e, (0,mx-len(e))) for e in f])

def hybrid_desc(c): return np.concatenate([fourier_desc(c,64), wavelet_desc(c)])

def zernike_moments(b, order=10):
    h,w = b.shape
    yg,xg = np.mgrid[-1:1:1j*h, -1:1:1j*w]
    r = np.sqrt(xg**2+yg**2); theta = np.arctan2(yg,xg)
    mask = (r<=1.) & (b>0); moms = []
    for n in range(order+1):
        for m in range(-n, n+1, 2):
            if (n-abs(m))%2 != 0: continue
            R = np.zeros_like(r)
            for s in range((n-abs(m))//2+1):
                c = ((-1)**s * scipy.special.factorial(n-s)) / (
                    scipy.special.factorial(s) *
                    scipy.special.factorial((n+abs(m))//2-s) *
                    scipy.special.factorial((n-abs(m))//2-s) + 1e-300)
                R += c * r**(n-2*s)
            V = R * np.exp(-1j*m*theta)
            moms.append(np.abs(np.sum(V[mask]*b[mask])*(n+1)/np.pi))
    return np.array(moms[:36])

def shape_context(c, nr=5, nt=12):
    N = len(c); step = max(1, N//64); pts = c[::step]; n = len(pts)
    dx = pts[:,1:2]-pts[np.newaxis,:,1]; dy = pts[:,0:1]-pts[np.newaxis,:,0]
    dist = np.sqrt(dx**2+dy**2+1e-12); ang = np.arctan2(dy, dx)
    ld = np.log(dist/(dist.max()+1e-12)+1e-12)
    rb = np.linspace(ld.min()-0.01, 0.01, nr+1)
    tb = np.linspace(-np.pi, np.pi, nt+1)
    Hg = np.zeros(nr*nt)
    for i in range(n):
        mi = np.arange(n)!=i
        H,_,_ = np.histogram2d(ld[i,mi], ang[i,mi], bins=[rb,tb])
        Hg += H.flatten()
    return Hg/(Hg.sum()+1e-12)

def css_desc(c, sigmas=None):
    if sigmas is None: sigmas = [1,2,4,8,16,32,64,128,256,512]
    x,yc = c[:,1], c[:,0]; f = []
    for s in sigmas:
        xs = ndimage.gaussian_filter1d(x, s, mode='wrap')
        ys = ndimage.gaussian_filter1d(yc, s, mode='wrap')
        x1=np.gradient(xs); x2=np.gradient(x1); y1=np.gradient(ys); y2=np.gradient(y1)
        k = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
        f += [float(np.sum(np.diff(np.sign(k))!=0)), float(np.mean(np.abs(k)))]
    return np.array(f)

def hog_desc(b):
    return feature.hog(b.astype(np.float32), orientations=9,
                       pixels_per_cell=(8,8), cells_per_block=(1,1), feature_vector=True)

print('Baselines defined.')
print('HOG dim:', len(hog_desc(bins[0])))

In [ ]:
# Cell 6: AMST v12 Descriptor — FIXED Components (C3 expanded to d=22)
# C1(144) + C2(90) + C3(253) + C5(38) = 525

# ── C1: APCFW+ ──────────────────────────────
def c1_apcfw_plus(c, K_F=64, n_wb=32):
    r = np.sqrt((c**2).sum(axis=1))
    F = np.fft.fft(r); mag = np.abs(F); ph = np.angle(F)
    denom = mag[1] if mag[1]>1e-8 else mag.max()+1e-12
    mag_n = mag / denom
    fd = mag_n[1:K_F+1]; phs = ph[1:17]
    n_star = int(np.argmax(mag[1:K_F+1])) + 1
    rho = n_star / K_F
    wv = 'db4' if rho < 0.10 else ('db2' if rho < 0.25 else 'haar')
    x,yc = c[:,1], c[:,0]
    x1=np.gradient(x); y1=np.gradient(yc); x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    kc = kappa - kappa.mean()
    max_lv = pywt.dwt_max_level(len(kc), wv)
    L = max(2, min(5, max_lv))
    try:
        coeffs = pywt.wavedec(kc, wv, level=L, mode='periodization')
    except:
        coeffs = pywt.wavedec(kc, 'db1', level=2, mode='periodization')
    energies = np.array([np.sum(cf**2) for cf in coeffs])
    E = energies / (energies.sum()+1e-12); n_act = len(E)
    h_idx = np.array([min(int(l * max(1, n_star/max(n_act,1))), K_F-1) for l in range(n_act)])
    h_idx = np.clip(h_idx, 0, len(ph)-1)
    cos_ph = np.abs(np.cos(ph[h_idx]))
    Omega = E * cos_ph + 1e-12; Omega /= Omega.sum()
    xi = np.linspace(0, 1, n_act); xo = np.linspace(0, 1, n_wb)
    Omega32 = interp1d(xi, Omega, kind='linear', fill_value='extrapolate')(xo)
    Omega32 = np.maximum(Omega32, 0); Omega32 /= Omega32.sum()+1e-12
    angles = np.arctan2(c[:,0], c[:,1])
    ang_hist,_ = np.histogram(angles, bins=32, range=(-np.pi, np.pi))
    ang_dist = ang_hist / (ang_hist.sum()+1e-12)
    return np.concatenate([fd, phs, Omega32, ang_dist])  # 64+16+32+32=144

# ── C2: Topology (FIXED: autocorrelation tau, robust filtering) ────
def c2_topology(c):
    x, yc = c[:,1], c[:,0]
    x1=np.gradient(x); y1=np.gradient(yc); x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    kn = (kappa - kappa.mean()) / (kappa.std() + 1e-12)
    N = len(kn)
    # Adaptive tau via autocorrelation
    ac = np.correlate(kn - kn.mean(), kn - kn.mean(), mode='full')[N-1:]
    ac /= ac[0] + 1e-12
    crossings = np.where(ac < 0.5)[0]
    tau = max(2, min(12, int(crossings[0]) if len(crossings) > 0 else N//15))
    if N < tau*3 + 3:
        return np.zeros(30)
    Xk = np.column_stack([kn[:N-tau], kn[tau:]])
    if len(Xk) > 200:
        idxs = np.linspace(0, len(Xk)-1, 200, dtype=int); Xk = Xk[idxs]
    try:
        dgms = ripser_fn(Xk, maxdim=1, n_perm=min(200, len(Xk)))['dgms']
    except:
        return np.zeros(30)

    def vec(dgm, k=7):
        fin = dgm[dgm[:,1] < np.inf]
        if len(fin) < 2:
            return np.zeros(k), np.zeros(k), np.zeros(6)
        lt = np.sort(fin[:,1]-fin[:,0])[::-1]
        bt = np.sort(fin[:,0])
        lt = lt[lt > np.percentile(lt, 25)]
        if len(lt) == 0: lt = np.array([0.0])
        bt = bt[:len(lt)]
        lp = np.zeros(k); lp[:min(len(lt),k)] = lt[:k]
        bp = np.zeros(k); bp[:min(len(bt),k)] = bt[:k]
        tot = lt.sum(); mx = lt[0] if len(lt)>0 else 0
        betti = float((lt > 0.1*tot).sum()) if tot>0 else 0
        ent = -np.sum(lt/(tot+1e-12)*np.log(lt/(tot+1e-12)+1e-12)) if tot>0 else 0
        med = float(np.median(lt)) if len(lt)>0 else 0
        return lp, bp, np.array([tot, mx, betti, ent, med, float(len(fin))])

    lt0,bt0,st0 = vec(dgms[0]); lt1,bt1,st1 = vec(dgms[1])
    feat = np.concatenate([lt0, lt1, bt0[:4], bt1[:4], st0, st1[:4]])
    out = np.zeros(30); out[:min(len(feat),30)] = feat[:30]
    return out  # 30

# ── C3: SPD Manifold (d=22, richer basis) ───────────
def c3_spd(c):
    r = np.sqrt((c**2).sum(axis=1)); N=len(r)
    x,yc = c[:,1], c[:,0]
    x1=np.gradient(x); y1=np.gradient(yc); x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    t = np.linspace(0, 2*np.pi, N, endpoint=False)
    rows = [
        r-r.mean(), x-x.mean(), yc-yc.mean(), kappa, np.cos(t), np.sin(t),
        ndimage.gaussian_filter1d(r-r.mean(),2,mode='wrap'),
        ndimage.gaussian_filter1d(r-r.mean(),8,mode='wrap'),
        ndimage.gaussian_filter1d(kappa,2,mode='wrap'),
        ndimage.gaussian_filter1d(kappa,4,mode='wrap'),
        ndimage.gaussian_filter1d(kappa,8,mode='wrap'),
        np.gradient(kappa),
        ndimage.gaussian_filter1d(r-r.mean(),4,mode='wrap'),
        ndimage.gaussian_filter1d(kappa,1,mode='wrap'),
        np.abs(kappa), kappa**2, np.sqrt(np.abs(kappa)+1e-12),
        np.sin(2*t), np.cos(2*t), np.arctan2(yc,x),
        ndimage.gaussian_filter1d(np.abs(kappa),2,mode='wrap'),
        np.sin(3*t)
    ]
    d=22; fm = np.array(rows[:d], dtype=float)
    fm -= fm.mean(axis=1, keepdims=True)
    norms = np.linalg.norm(fm, axis=1, keepdims=True)
    fm /= (norms+1e-12)
    S = (fm @ fm.T)/(N-1) + 1e-4*np.eye(d)
    ev,evec = np.linalg.eigh(S)
    ev = np.maximum(ev, 1e-8)
    logS = evec @ np.diag(np.log(ev)) @ evec.T
    return logS[np.triu_indices(d)]  # 253

# ── C5: Complexity (expanded) ────────────────
def c5_complexity(c, b):
    f = []
    hull = ConvexHull(c)
    ha = hull.volume; hp = hull.area
    ca = np.abs(np.sum(c[:-1,0]*c[1:,1]-c[1:,0]*c[:-1,1]))/2
    cp = np.sum(np.sqrt(np.diff(c[:,0])**2+np.diff(c[:,1])**2))
    f += [hp/(cp+1e-12), ca/(ha+1e-12), 4*np.pi*ca/(cp**2+1e-12)]
    labeled = measure.label(b); props = regionprops(labeled)
    if props:
        p=props[0]; f += [float(p.euler_number), p.major_axis_length/(p.minor_axis_length+1e-12),
                          p.extent, p.eccentricity, p.equivalent_diameter_area/max(b.shape),
                          p.perimeter/(cp+1e-12), p.area/(b.shape[0]*b.shape[1])]
    else: f += [0.0]*7
    x,yc=c[:,1],c[:,0]
    x1=np.gradient(x); y1=np.gradient(yc); x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    f += [np.mean(kappa), np.std(kappa), np.max(np.abs(kappa)),
          np.sum(np.diff(np.sign(kappa))!=0)/len(kappa),
          float(scipy.stats.skew(kappa)), float(scipy.stats.kurtosis(kappa)),
          float(np.percentile(np.abs(kappa), 90))]
    d = np.sqrt((c**2).sum(axis=1))
    f += [np.std(d), float(np.max(d)-np.min(d)), np.mean(d),
          float(np.percentile(d,25)), float(np.percentile(d,75))]
    # Additional shape stats
    f += [float(np.percentile(d,10)), float(np.percentile(d,90))]
    f += [np.sum(kappa > 0)/len(kappa), np.sum(kappa < 0)/len(kappa)]
    f += [np.max(d)/np.min(d+1e-12), np.sqrt(np.mean(d**2))]
    f += [float(scipy.stats.variation(d+1e-12))]
    return np.array(f, dtype=float)  # 38

# ── C4: Attention (FIXED - deterministic, stratified) ───────
class MHFisherAttn:
    def __init__(self, n_heads=8, n_bands=16, top_k=0.5, seed=42):
        self.nh=n_heads; self.nb=n_bands; self.tk=top_k
        self.w=None; self.bb=None; self.seed=seed
    def fit(self, X, y_tr):
        N,D=X.shape; bs=max(1, D//self.nb)
        self.bb=[(i*bs, min((i+1)*bs,D)) for i in range(self.nb)]
        cls=np.unique(y_tr); k=max(1,int(self.nb*self.tk))
        hw=[]
        rng = np.random.RandomState(self.seed)
        for h in range(self.nh):
            sub = []
            for c in cls:
                idx_c = np.where(y_tr==c)[0]
                n_c = max(1, int(0.8*len(idx_c)))
                sub.extend(rng.choice(idx_c, n_c, replace=False).tolist())
            Xs, ys = X[sub], y_tr[sub]; gms = Xs.mean(0); scs=[]
            for b0,b1 in self.bb:
                Xb=Xs[:,b0:b1]; gmb=gms[b0:b1]; SB=SW=0.0
                for c in cls:
                    mk=ys==c
                    if mk.sum()<2: continue
                    mc=Xb[mk].mean(0)
                    SB+=mk.sum()*np.dot(mc-gmb,mc-gmb)
                    SW+=np.sum((Xb[mk]-mc)**2)
                scs.append(SB/(SW+1e-8))
            sc=np.array(scs)
            sc_z = (sc - sc.mean()) / (sc.std() + 1e-8)
            sc_sm = np.exp(sc_z * 0.5)
            sc_sm /= sc_sm.sum() + 1e-12
            sp=np.zeros(self.nb)
            top=np.argsort(sc_sm)[::-1][:k]
            sp[top]=sc_sm[top]
            sp/=sp.sum()+1e-12
            hw.append(sp)
        avg=np.mean(hw,0); avg/=avg.sum()+1e-12
        self.w=np.zeros(D)
        for i,(b0,b1) in enumerate(self.bb): self.w[b0:b1]=avg[i]
        return self
    def transform(self,X):
        if self.w is None: raise ValueError('fit first')
        return X*self.w[np.newaxis,:]

# ── Full AMST v12 ──────────────────────
def amst_v12_raw(c, b):
    return (c1_apcfw_plus(c), c2_topology(c), c3_spd(c), c5_complexity(c, b))

def amst_v12(c, b, c1_m=None, c1_s=None, c2_m=None, c2_s=None, c3_m=None, c3_s=None, c5_m=None, c5_s=None):
    c1, c2, c3, c5 = amst_v12_raw(c, b)
    if c1_m is not None:
        c1 = (c1 - c1_m) / (c1_s + 1e-12)
        c2 = (c2 - c2_m) / (c2_s + 1e-12)
        c3 = (c3 - c3_m) / (c3_s + 1e-12)
        c5 = (c5 - c5_m) / (c5_s + 1e-12)
    return np.concatenate([c1, c2, c3, c5])

tst = amst_v12(cnts[0], bins[0])
c1_len = len(c1_apcfw_plus(cnts[0]))
c2_len = len(c2_topology(cnts[0]))
c3_len = len(c3_spd(cnts[0]))
c5_len = len(c5_complexity(cnts[0],bins[0]))
print(f'C1:{c1_len} C2:{c2_len} C3:{c3_len} C5:{c5_len} Total:{c1_len+c2_len+c3_len+c5_len}')
print(f'Total: {len(tst)} (expected {AMST_DIM})')
assert len(tst)==AMST_DIM, f'Dim mismatch: {len(tst)} != {AMST_DIM}'
print('AMST v12 OK')

In [ ]:
# Cell 7: Feature Extraction
print('Extracting features...')
N = len(cnts)
FD, WD, HY, ZE, CS, SC, HG = [],[],[],[],[],[],[]
AM_C1, AM_C2, AM_C3, AM_C5 = [],[],[],[]
for i in tqdm(range(N)):
    c=cnts[i]; b=bins[i]
    try: FD.append(fourier_desc(c))
    except: FD.append(np.zeros(79))
    try: WD.append(wavelet_desc(c))
    except: WD.append(np.zeros(18))
    try: HY.append(hybrid_desc(c))
    except: HY.append(np.zeros(97))
    try: ZE.append(zernike_moments(b))
    except: ZE.append(np.zeros(36))
    try: CS.append(css_desc(c))
    except: CS.append(np.zeros(20))
    try: SC.append(shape_context(c))
    except: SC.append(np.zeros(60))
    try: HG.append(hog_desc(b))
    except: HG.append(np.zeros(576))
    try:
        c1,c2,c3,c5 = amst_v12_raw(c,b)
        AM_C1.append(c1); AM_C2.append(c2); AM_C3.append(c3); AM_C5.append(c5)
    except:
        AM_C1.append(np.zeros(144)); AM_C2.append(np.zeros(90))
        AM_C3.append(np.zeros(253)); AM_C5.append(np.zeros(38))

X_fd=np.array(FD); X_wd=np.array(WD); X_hy=np.array(HY)
X_ze=np.array(ZE); X_cs=np.array(CS); X_sc=np.array(SC)
X_hg=np.array(HG)
AM_c1=np.array(AM_C1); AM_c2=np.array(AM_C2); AM_c3=np.array(AM_C3); AM_c5=np.array(AM_C5)

# Standardize each AMST component separately
c1_m,c1_s=AM_c1.mean(0),AM_c1.std(0); c1_s=np.clip(c1_s,1e-12,None)
c2_m,c2_s=AM_c2.mean(0),AM_c2.std(0); c2_s=np.clip(c2_s,1e-12,None)
c3_m,c3_s=AM_c3.mean(0),AM_c3.std(0); c3_s=np.clip(c3_s,1e-12,None)
c5_m,c5_s=AM_c5.mean(0),AM_c5.std(0); c5_s=np.clip(c5_s,1e-12,None)
AM_c1_s = (AM_c1-c1_m)/c1_s
AM_c2_s = (AM_c2-c2_m)/c2_s
AM_c3_s = (AM_c3-c3_m)/c3_s
AM_c5_s = (AM_c5-c5_m)/c5_s
X_am = np.nan_to_num(np.concatenate([AM_c1_s, AM_c2_s, AM_c3_s, AM_c5_s], axis=1))

for nm,X in [('FD',X_fd),('WD',X_wd),('HY',X_hy),('ZE',X_ze),
             ('CS',X_cs),('SC',X_sc),('HG',X_hg),('AM',X_am)]:
    print(f'  {nm:4s}: {X.shape}')
assert X_am.shape[1]==AMST_DIM
print('Features done.')
gc.collect()

In [ ]:
# Cell 8: Evaluation (5-fold CV, expanded SVM grid, attention only for AMST)
def best_svm(X_tr, y_tr):
    param_grid = {
        'C': [0.01, 0.1, 1, 10, 100, 1000, 10000],
        'gamma': ['scale', 'auto', 1, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0005]
    }
    gs = GridSearchCV(SVC(kernel='rbf', decision_function_shape='ovr', class_weight='balanced'),
                      param_grid, cv=min(3, len(np.unique(y_tr))),
                      scoring='accuracy', n_jobs=-1, verbose=0)
    gs.fit(X_tr, y_tr); return gs.best_params_

def eval_svm(X, y, name, attn=False):
    X = np.nan_to_num(X.copy())
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    accs=[]; f1s=[]; faccs=[]
    for tr, te in skf.split(X, y):
        X_tr,X_te=X[tr],X[te]; y_tr,y_te=y[tr],y[te]
        if attn:
            a=MHFisherAttn(seed=SEED); a.fit(X_tr,y_tr); X_tr=a.transform(X_tr); X_te=a.transform(X_te)
        sc=RobustScaler(); X_tr_s=sc.fit_transform(X_tr); X_te_s=sc.transform(X_te)
        bp=best_svm(X_tr_s, y_tr)
        clf=SVC(kernel='rbf', decision_function_shape='ovr', class_weight='balanced', **bp)
        clf.fit(X_tr_s,y_tr); yp=clf.predict(X_te_s)
        accs.append(accuracy_score(y_te,yp))
        f1s.append(f1_score(y_te,yp,average='macro',zero_division=0))
        faccs.append(accuracy_score(y_te,yp))
    return {'Method':name, 'Accuracy':np.mean(accs), 'Acc_std':np.std(accs),
            'F1':np.mean(f1s), 'Dim':X.shape[1], 'faccs':faccs}

def stack_pred(X_tr, y_tr, X_te):
    rf = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=SEED, n_jobs=-1)
    xgb_c = xgb.XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=SEED, n_jobs=-1, verbosity=0,
                               eval_metric='mlogloss') if XGB_OK else None
    rf_p = cross_val_predict(rf, X_tr, y_tr, cv=3, method='predict_proba')
    if xgb_c is not None:
        xgb_p = cross_val_predict(xgb_c, X_tr, y_tr, cv=3, method='predict_proba')
        mt = np.column_stack([rf_p, xgb_p])
    else: mt = rf_p
    rf.fit(X_tr, y_tr); rf_t = rf.predict_proba(X_te)
    if xgb_c is not None:
        xgb_c.fit(X_tr, y_tr); mt_t = np.column_stack([rf_t, xgb_c.predict_proba(X_te)])
    else: mt_t = rf_t
    meta = LogisticRegression(multi_class='multinomial', max_iter=3000, C=0.5, random_state=SEED)
    meta.fit(mt, y_tr); return meta.predict(mt_t)

def eval_stack(X, y, name, attn=False):
    X = np.nan_to_num(X.copy())
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    accs=[]; f1s=[]
    for tr, te in skf.split(X, y):
        X_tr,X_te=X[tr],X[te]; y_tr,y_te=y[tr],y[te]
        if attn:
            a=MHFisherAttn(seed=SEED); a.fit(X_tr,y_tr); X_tr=a.transform(X_tr); X_te=a.transform(X_te)
        sc=RobustScaler(); X_tr_s=sc.fit_transform(X_tr); X_te_s=sc.transform(X_te)
        yp=stack_pred(X_tr_s, y_tr, X_te_s)
        accs.append(accuracy_score(y_te,yp))
        f1s.append(f1_score(y_te,yp,average='macro',zero_division=0))
    return {'Method':name, 'Accuracy':np.mean(accs), 'Acc_std':np.std(accs), 'F1':np.mean(f1s)}

print('Evaluation ready.')

In [ ]:
# Cell 9: 5-Fold CV (AMST gets attention, baselines do not)
print('Running 5-fold CV...\n')
svm_res = []
for Xf, nm, attn in [(X_fd,'Fourier',0),(X_wd,'Wavelet',0),(X_hy,'Hybrid',0),
                     (X_ze,'Zernike',0),(X_cs,'CSS',0),(X_sc,'ShapeCtx',0),
                     (X_hg,'HOG',0),(X_am,'AMST',1)]:
    r = eval_svm(Xf, y, nm, attn)
    svm_res.append(r)
    print(f'{nm:12s} | Acc:{r["Accuracy"]*100:.2f}% F1:{r["F1"]*100:.2f}%')

df = pd.DataFrame(svm_res)
amst_acc = df[df.Method=='AMST']['Accuracy'].values[0]*100
best_base = df[df.Method!='AMST']['Accuracy'].max()*100
print(f'\nAMST: {amst_acc:.2f}% | Best base: {best_base:.2f}% | Gain: +{amst_acc-best_base:.2f} pp')

print('\nAMST stacking...')
sr = eval_stack(X_am, y, 'AMST+Stack', attn=True)
amst_stk = sr['Accuracy']*100
print(f'AMST+Stack: {amst_stk:.2f}% (gain over SVM: +{amst_stk-amst_acc:.2f} pp)')
svm_res.append(sr)

print('\nAll results:')
print(pd.DataFrame(svm_res)[['Method','Accuracy','F1','Dim']].to_string(index=False))

In [ ]:
# Cell 10: Significance (Bonferroni)
amst_f = np.array(df[df.Method=='AMST']['faccs'].values[0])
N_CMP = len(df)-1
ALPHA_BONF = 0.05/N_CMP

print(f'Bonferroni: {N_CMP} comparisons, alpha={ALPHA_BONF:.6f}')
print(f'{"Method":<14} {"AMST%":>8} {"Base%":>8} {"Delta":>8} {"p-val":>10} {"Sig?":>8}')
print('-'*50)

rows = []
for _, r in df.iterrows():
    nm = r.Method
    if nm=='AMST': continue
    bf = np.array(r['faccs'])
    t, p = scipy.stats.ttest_rel(amst_f, bf)
    d = (amst_f.mean()-bf.mean())*100
    sb = '*' if p<ALPHA_BONF else ' '
    print(f'{nm:14s} {amst_f.mean()*100:>8.2f} {bf.mean()*100:>8.2f} {d:>+8.2f} {p:>10.5f} {sb:>8}')
    rows.append({'Baseline':nm, 'Delta_pp':d, 'p_value':p, 'Bonf_sig':p<ALPHA_BONF})

stat_df = pd.DataFrame(rows)
print(f'\nSig wins (Bonferroni): {stat_df.Bonf_sig.sum()}/{N_CMP}')

In [ ]:
# Cell 11: Ablation (5-fold, SVM, attention)
def abl_var(c, b, v):
    c1=c1_apcfw_plus(c); c2=c2_topology(c); c3=c3_spd(c); c5=c5_complexity(c,b)
    if v=='c1': return c1
    elif v=='c12': return np.concatenate([c1,c2])
    elif v=='c123': return np.concatenate([c1,c2,c3])
    else: return np.concatenate([c1,c2,c3,c5])

print('Ablation...')
vmap = [('c1','C1(144)',144),('c12','C1+C2(234)',234),('c123','C1+C2+C3(487)',487),('full','Full(525)',525)]
abl_f = {}
for vn,vl,ed in vmap:
    ff=[]
    for i in tqdm(range(N), desc=vl, leave=False):
        try: ff.append(abl_var(cnts[i],bins[i],vn))
        except: ff.append(np.zeros(ed))
    abl_f[vl]=np.nan_to_num(np.array(ff))

abl_res=[]
for vl,Xa in abl_f.items():
    r=eval_svm(Xa,y,vl,attn=True); abl_res.append(r)
    print(f'{vl:25s} Acc:{r["Accuracy"]*100:.2f}%')

abl_df=pd.DataFrame(abl_res)
abl_folds=[np.array(r['faccs']) for r in abl_res]
for i in range(1,len(abl_folds)):
    t,p=scipy.stats.ttest_rel(abl_folds[i],abl_folds[i-1])
    d=(abl_folds[i].mean()-abl_folds[i-1].mean())*100
    print(f'  {abl_res[i-1]["Method"][:20]:>20} -> {abl_res[i]["Method"][:20]:20} {d:+>7.2f}pp p={p:.4f}')

In [ ]:
# Figure 2: Ablation
fig,ax=plt.subplots(figsize=(10,4))
names=[r['Method'] for r in abl_res]
accs=[r['Accuracy']*100 for r in abl_res]
stds=[r['Acc_std']*100 for r in abl_res]
colors=['#AED6F1','#5DADE2','#2471A3','#1A5276','#E84040']
bars=ax.barh(names,accs,xerr=stds,color=colors,capstyle=4,height=0.6)
for b,ac in zip(bars,accs):
    ax.text(ac+0.3,b.get_y()+b.get_height()/2,f'{ac:.1f}%',va='center',fontsize=9)
ax.set_xlim(0,105); ax.set_xlabel('Accuracy (%)'); ax.set_title('Ablation: Incremental AMST Components')
ax.grid(axis='x',alpha=0.3); plt.tight_layout(); plt.savefig('/content/fig2_ablation.png',dpi=120,bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Classification
fig,axes=plt.subplots(1,2,figsize=(14,5))
methods=df.Method.values
accs=df.Accuracy.values*100; stds=df.Acc_std.values*100; f1s=df.F1.values*100
colors=['#5B7FA6']*7+['#E84040']
for ax,vals,title in [(axes[0],accs,'Accuracy'),(axes[1],f1s,'F1')]:
    bars=ax.barh(methods,vals,xerr=stds if ax==axes[0] else None,
                 color=colors,edgecolor='white',capsize=4,height=0.65)
    ax.set_xlim(0,110); ax.set_xlabel('%'); ax.set_title(title); ax.grid(axis='x',alpha=0.3)
    for i,(b,v) in enumerate(zip(bars,vals)):
        fw='bold' if i==len(methods)-1 else 'normal'
        ax.text(v+0.5,b.get_y()+b.get_height()/2,f'{v:.1f}%',va='center',fontsize=8,fontweight=fw)
plt.tight_layout(); plt.savefig('/content/fig3_classification.png',dpi=120,bbox_inches='tight')
plt.show(); print('Fig 3 saved.')

In [ ]:
# Figure 4: Confusion (AMST)
skf=StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED)
all_yt=[]; all_yp=[]
for tr,te in skf.split(X_am,y):
    X_tr,X_te=X_am[tr],X_am[te]; y_tr,y_te=y[tr],y[te]
    a=MHFisherAttn(seed=SEED); a.fit(X_tr,y_tr); X_tr=a.transform(X_tr); X_te=a.transform(X_te)
    sc=RobustScaler(); X_tr_s=sc.fit_transform(X_tr); X_te_s=sc.transform(X_te)
    bp=best_svm(X_tr_s,y_tr)
    clf=SVC(kernel='rbf',decision_function_shape='ovr',class_weight='balanced',**bp)
    clf.fit(X_tr_s,y_tr)
    all_yp.extend(clf.predict(X_te_s)); all_yt.extend(y_te)

cm_acc=accuracy_score(all_yt,all_yp)*100
cm=confusion_matrix(all_yt,all_yp)
cm_pct=cm.astype(float)/cm.sum(axis=1,keepdims=True)*100
print(f'CV agg acc: {cm_acc:.2f}%')
diag=np.diag(cm_pct)
worst=np.argsort(diag)[:5]
print('Worst classes:')
for w in worst: print(f'  {le.classes_[w].capitalize():15s}: {diag[w]:.1f}%')

fig,ax=plt.subplots(figsize=(8,7))
im=ax.imshow(cm_pct,cmap='Blues',vmin=0,vmax=100)
plt.colorbar(im,ax=ax,label='%')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'AMST v12 ({cm_acc:.1f}%)')
plt.tight_layout(); plt.savefig('/content/fig4_confusion.png',dpi=120,bbox_inches='tight')
plt.show()

In [ ]:
# Cell 12: Retrieval (Bullseye Rating + MAP)
def bullseye(X, y, K=40):
    Xs=StandardScaler().fit_transform(np.nan_to_num(X))
    N=len(y); good=0; poss=0
    for qi in range(N):
        d=np.sqrt(((Xs-Xs[qi])**2).sum(1))
        rk=np.argsort(d); rk=rk[rk!=qi]
        matches=(y[rk[:K]]==y[qi]).sum()
        n_same=(y==y[qi]).sum()-1
        good+=matches; poss+=min(K,n_same)
    return good/poss*100

def map_score(X, y):
    Xs=StandardScaler().fit_transform(np.nan_to_num(X))
    N=len(y); APs=[]
    for qi in range(N):
        d=np.sqrt(((Xs-Xs[qi])**2).sum(1))
        rk=np.argsort(d); rk=rk[rk!=qi]
        rel=(y[rk]==y[qi]).astype(int)
        if rel.sum()==0: continue
        cs=np.cumsum(rel); pos=np.arange(1,len(rk)+1)
        APs.append((cs/pos*rel).sum()/rel.sum())
    return np.mean(APs)

print('Retrieval...')
ret_m = [(X_fd,'Fourier'),(X_wd,'Wavelet'),(X_hy,'Hybrid'),
         (X_ze,'Zernike'),(X_sc,'ShapeCtx'),(X_am,'AMST')]
ret_r=[]
for Xf,nm in ret_m:
    be=bullseye(Xf,y,40); mp=map_score(Xf,y)
    ret_r.append({'Method':nm,'Bullseye':be,'MAP':mp})
    print(f'{nm:12s} Bullseye:{be:.1f}% MAP:{mp:.4f}')

rdf=pd.DataFrame(ret_r)
print(f'\nAMST Bullseye: {rdf[rdf.Method=="AMST"]["Bullseye"].values[0]:.1f}%')
print(f'Best: {rdf.Bullseye.max():.1f}%')
print(rdf.to_string(index=False))

In [ ]:
# Figure 5: PR Curves
def pr_curve(X,y):
    Xs=StandardScaler().fit_transform(np.nan_to_num(X))
    N=len(y); APs=[]; allP=[]; allR=[]
    for qi in range(N):
        d=np.sqrt(((Xs-Xs[qi])**2).sum(1))
        rk=np.argsort(d)[1:]
        rel=(y[rk]==y[qi]).astype(int)
        if rel.sum()==0: continue
        cs=np.cumsum(rel); pos=np.arange(1,len(rk)+1)
        APs.append((cs/pos*rel).sum()/rel.sum())
        allP.append(cs/pos); allR.append(cs/rel.sum())
    rg=np.linspace(0,1,20)
    ip=[np.interp(rg,r,p) for p,r in zip(allP,allR)]
    return rg,np.mean(ip,0),np.mean(APs)

cpr=['#5B7FA6','#E8A020','#27AE60','#8E44AD','#34495E','#E84040']
pr_m=[(X_fd,'Fourier'),(X_wd,'Wavelet'),(X_hy,'Hybrid'),(X_ze,'Zernike'),(X_sc,'ShapeCtx'),(X_am,'AMST')]
curves={}
for (Xf,nm),col in zip(pr_m,cpr):
    rg,mp,ma=pr_curve(Xf,y); curves[nm]=(rg,mp,ma)

fig,ax=plt.subplots(figsize=(9,6))
for (nm,(rc,pr,ma)),col,lw in zip(curves.items(),cpr,[1.5]*5+[2.8]):
    ls='-' if 'AMST' in nm else '--'
    ax.plot(rc,pr,color=col,lw=lw,label=f'{nm}(MAP={ma:.3f})',ls=ls)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('Retrieval PR')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_xlim(0,1); ax.set_ylim(0,1.05)
plt.tight_layout(); plt.savefig('/content/fig5_pr.png',dpi=120,bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6: Dashboard
fig=plt.figure(figsize=(16,10))
fig.suptitle(f'AMST v12 FIXED — MPEG-7 CE-Shape-1 | {AMST_DIM}d | Bonferroni | {N_SPLITS}-fold',fontsize=13,fontweight='bold',y=1.01)

ax1=fig.add_subplot(2,3,1)
ax1.barh(methods,accs,color=colors,edgecolor='white',height=0.65)
ax1.set_title('Accuracy'); ax1.set_xlim(0,110); ax1.grid(axis='x',alpha=0.3)

ax2=fig.add_subplot(2,3,2)
rdf_s=rdf.sort_values('Bullseye')
ax2.barh(rdf_s['Method'],rdf_s['Bullseye'],
         color=['#E84040' if 'AMST' in n else '#5B7FA6' for n in rdf_s['Method']])
ax2.set_title('Bullseye Rating (%)'); ax2.set_xlim(0,105); ax2.grid(axis='x',alpha=0.3)

ax3=fig.add_subplot(2,3,3)
an=[r['Method'][:20] for r in abl_res]; aa=[r['Accuracy']*100 for r in abl_res]
ax3.barh(an,aa,color=['#AED6F1','#5DADE2','#2471A3','#1A5276','#E84040'])
ax3.set_title('Ablation'); ax3.set_xlim(0,105); ax3.grid(axis='x',alpha=0.3)

ax4=fig.add_subplot(2,3,4)
st=stat_df.sort_values('Delta_pp')
ax4.barh(st['Baseline'],st['Delta_pp'],
         color=['#27AE60' if s else '#E74C3C' for s in st['Bonf_sig']])
ax4.axvline(0,color='k',lw=1); ax4.set_title('Gain over Baselines (pp)'); ax4.grid(axis='x',alpha=0.3)

ax5=fig.add_subplot(2,3,5); ax5.axis('off')
txt=(f'MPEG-7: {len(cnts)} imgs, {n_cls} cls\nAMST: {amst_acc:.1f}%\nBest base: {best_base:.1f}%\n'
     f'Gain: +{amst_acc-best_base:.1f}pp\nStack: {amst_stk:.1f}%\nBonf sig: {stat_df.Bonf_sig.sum()}/{N_CMP}\n'
     f'Bullseye: {rdf[rdf.Method=="AMST"]["Bullseye"].values[0]:.1f}%')
ax5.text(0.05,0.5,txt,fontsize=10,fontfamily='monospace',va='center',
         bbox=dict(boxstyle='round',facecolor='lightyellow',alpha=0.8))

plt.tight_layout(); plt.savefig('/content/fig6_dashboard.png',dpi=120,bbox_inches='tight')
plt.show(); print('Fig 6 saved.')

In [ ]:
# Save
import os
out='/content/amst_v12_results'; os.makedirs(out,exist_ok=True)
df[['Method','Accuracy','Acc_std','F1','Dim']].to_csv(f'{out}/results.csv',index=False)
stat_df.to_csv(f'{out}/significance.csv',index=False)
pd.DataFrame(abl_res).to_csv(f'{out}/ablation.csv',index=False)
rdf.to_csv(f'{out}/retrieval.csv',index=False)
print('Saved to',out)
print('=== AMST v12 FIXED Complete — MPEG-7 CE-Shape-1 ===')